In [91]:
import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = "1"
from joblib import Parallel, delayed

import numpy as np

from dysts.maps import Henon

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, root_mean_squared_error

from spring_rc.v1 import (
    build_von_mises_lattice,
    run_simulation,
    scatter_plot,
    save_figure,
    r_2_plots_grid,
)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [92]:
steps = 1000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [93]:
# henon_model = Henon()
# henon_dataset = henon_model.make_trajectory(total_steps)
# henon_dataset = henon_dataset[transient_steps_chaos:]

# henon_scaler = StandardScaler()
# henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
# henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
# henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

noise_data = np.random.uniform(
    low=0.0, high=1.0, size=(total_steps - transient_steps_chaos)
)

# Noise Energy and Springs

In [98]:
rng = np.random.default_rng(42)

num_cells = 100
cell_spacing = 2.5
wall_dist = 1.0
mass_val = 0.01
damping_val = 0.2
k_wall_val = 5.0
k_inter_val = 4.0
rest_wall_ratio = 1.1
rest_inter_ratio = 1.1
k_wall_spread = 1.0
k_inter_spread = 1.0
rest_wall_ratio_spread = 0.0
rest_inter_ratio_spread = 0.0

dt = 0.01
target_node_count = 10
input_force = 10
free_steps = 3

N_step = int(num_cells / target_node_count)
target_nodes = 1 + (np.arange(target_node_count) * N_step) * 3

F_in = np.zeros((total_steps_after_chaos, target_node_count, 2))
for j in range(target_nodes.shape[0]):
    # F_in[::, j, 0] = noise_data[:, j % 2]
    F_in[::, j, 0] = noise_data
F_in = F_in * input_force

In [99]:
(
    init_pos,
    m_inv_diag,
    c_diag,
    connections_list,
    wall_nodes,
    k_vals,
    rest_lens,
) = build_von_mises_lattice(
    rng=rng,
    num_cells=num_cells,
    cell_spacing=cell_spacing,
    wall_dist=wall_dist,
    mass_val=mass_val,
    damping_val=damping_val,
    k_wall_val=k_wall_val,
    k_wall_spread=k_wall_spread,
    k_inter_val=k_inter_val,
    k_inter_spread=k_inter_spread,
    rest_wall_ratio=rest_wall_ratio,
    rest_wall_ratio_spread=rest_wall_ratio_spread,
    rest_inter_ratio=rest_inter_ratio,
    rest_inter_ratio_spread=rest_inter_ratio_spread,
)

Xd, _ = run_simulation(
    steps=total_steps_after_chaos,
    free_steps=free_steps,
    dt=dt,
    m_inv=m_inv_diag,
    c_diag=c_diag,
    F_in=F_in,
    target_nodes=target_nodes,
    init_pos=init_pos,
    conn=connections_list,
    k=k_vals,
    rest=rest_lens,
    fixed_nodes=wall_nodes,
    rec_nodes=1 + 3 * np.arange(num_cells),
)

In [101]:
X_data = Xd[transient_steps_reservoir:-tau_steps]
Y_data = noise_data[transient_steps_reservoir + tau_steps :]

X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)
Y_train, Y_test = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)
model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

In [102]:
rmse = root_mean_squared_error(Y_test, Y_pred, multioutput="raw_values")
nrmse = rmse / np.std(Y_test, axis=0)
r_2 = r2_score(Y_test, Y_pred, multioutput="raw_values")
nrmse, r_2

(array([1.00511126]), array([-0.01024864]))

In [103]:
henon_attractor_prediction = scatter_plot(
    data_list=[Y_pred, Y_test],
    data_names=["Predicted", "Actual"],
    axis_names=["X", "Y"],
    title="Henon Attractor",
)
henon_attractor_r2 = r_2_plots_grid(
    Y_pred, Y_test, r_2, subplot_titles=["Henon X", "Henon Y"], title="Henon R²"
)

henon_attractor_prediction.show()
henon_attractor_r2.show()

In [104]:
X_data[::, 10].shape

(1000,)

In [ ]:
data = 1 / 2 * X_data[::, 10] ** 2


def moving_average(values, window):
    weights = np.repeat(1.0, window) / window
    return np.convolve(values, weights, "valid")


window_size = 20
smooth_data = moving_average(data, window_size)

# Convouliton shirnks array length we ceneter again
x_raw = np.arange(data.shape[0])
x_smooth = np.arange(window_size // 2, window_size // 2 + smooth_data.shape[0])

fig = go.Figure()

# Raw
fig.add_trace(
    go.Scatter(
        x=x_raw,
        y=data,
        mode="lines",
        line=dict(width=1, color="rgba(0, 100, 255, 0.25)"),
        name="Raw Energy",
    )
)

# Smooth
fig.add_trace(
    go.Scatter(
        x=x_smooth,
        y=smooth_data,
        mode="lines",
        line=dict(width=2.5, color="rgb(200, 20, 20)"),
        name=f"Moving Average (w={window_size})",
    )
)

fig.update_layout(
    title="Spring Potential Energy (Smoothed Trend)",
    xaxis_title="Step / Time",
    yaxis_title="Potential Energy",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
)

fig.show()

# Variance Energy

In [169]:
rng = np.random.default_rng(42)

num_cells = 100
cell_spacing = 2.5
wall_dist = 1.0
mass_val = 0.01
damping_val = 0.2
k_wall_val = 5.0
k_inter_val = 4.0
rest_wall_ratio = 1.1
rest_inter_ratio = 1.1
k_wall_spread = 0.0
k_inter_spread = 0.0
rest_wall_ratio_spread = 0.0
rest_inter_ratio_spread = 0.0

dt = 0.01
target_node_count = 10
input_force = 10
free_steps = 3

N_step = int(num_cells / target_node_count)
target_nodes = 1 + (np.arange(target_node_count) * N_step) * 3

F_in = np.zeros((total_steps_after_chaos, target_node_count, 2))
for j in range(target_nodes.shape[0]):
    # F_in[::, j, 0] = noise_data[:, j % 2]
    F_in[::, j, 0] = noise_data
F_in = F_in * input_force

In [177]:
def run_trial(k_w_spread, k_i_spread, r_w_ratio, r_i_ratio, r_w_spread, r_i_spread_val):
    (
        init_pos,
        m_inv_diag,
        c_diag,
        connections_list,
        wall_nodes,
        k_vals,
        rest_lens,
    ) = build_von_mises_lattice(
        rng=rng,
        num_cells=num_cells,
        cell_spacing=cell_spacing,
        wall_dist=wall_dist,
        mass_val=mass_val,
        damping_val=damping_val,
        k_wall_val=k_wall_val,
        k_wall_spread=k_w_spread,
        k_inter_val=k_inter_val,
        k_inter_spread=k_i_spread,
        rest_wall_ratio=r_w_ratio,
        rest_wall_ratio_spread=r_w_spread,
        rest_inter_ratio=r_i_ratio,
        rest_inter_ratio_spread=r_i_spread_val,
    )

    Xd, _ = run_simulation(
        steps=total_steps_after_chaos,
        free_steps=free_steps,
        dt=dt,
        m_inv=m_inv_diag,
        c_diag=c_diag,
        F_in=F_in,
        target_nodes=target_nodes,
        init_pos=init_pos,
        conn=connections_list,
        k=k_vals,
        rest=rest_lens,
        fixed_nodes=wall_nodes,
        rec_nodes=1 + 3 * np.arange(num_cells),
    )

    X_data = Xd[transient_steps_reservoir:-tau_steps]
    Y_data = noise_data[transient_steps_reservoir + tau_steps :]
    X_train, X_test = (
        X_data[:-test_steps],
        X_data[-test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-test_steps],
        Y_data[-test_steps:],
    )
    model = RidgeCV()
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    return X_data

In [206]:
def plot_smoothed_energy(
    data, window_size=20, title="Spring Potential Energy (Smoothed Trend)"
):
    def moving_average(values, window):
        weights = np.repeat(1.0, window) / window
        return np.convolve(values, weights, "valid")

    smooth_data = moving_average(data, window_size)

    # Convouliton shirnks array length we ceneter again
    x_raw = np.arange(data.shape[0])
    x_smooth = np.arange(window_size // 2, window_size // 2 + smooth_data.shape[0])

    fig = go.Figure()

    # Raw
    fig.add_trace(
        go.Scatter(
            x=x_raw,
            y=data,
            mode="lines",
            line=dict(width=1, color="rgba(0, 100, 255, 0.25)"),
            name="Raw Energy",
        )
    )

    # Smooth
    fig.add_trace(
        go.Scatter(
            x=x_smooth,
            y=smooth_data,
            mode="lines",
            line=dict(width=2.5, color="rgb(200, 20, 20)"),
            name=f"Moving Average (w={window_size})",
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title="Step / Time",
        yaxis_title="Potential Energy",
        template="plotly_white",
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    )

    fig.show()

In [207]:
X_uniform = run_trial(0.0, 0.0, 1.0, 1.0, 0.0, 0.0)

X_stiffness = run_trial(1.0, 1.0, 1.0, 1.0, 0.0, 0.0)

X_ratio = run_trial(0.0, 0.0, 1.1, 1.1, 1.0, 1.0)

In [208]:
plot_smoothed_energy(0.5 * X_uniform[::, 10] ** 2, title="Uniform Case")

plot_smoothed_energy(0.5 * X_stiffness[::, 10] ** 2, title="Stiffness Spread Case")

plot_smoothed_energy(0.5 * X_ratio[::, 10] ** 2, title="Ratio Spread Case")

In [209]:
def plot_total_system_energy(
    X_data, window_size=20, title="Total Lattice Potential Energy"
):
    # 1. Calculate potential energy for every node/dimension across all time steps
    # (Assuming X_data contains position coordinates for all nodes)
    node_energies = 0.5 * (X_data**2)

    # 2. Sum the energy across all nodes for each time step to get total system energy
    total_energy = np.sum(node_energies, axis=1)

    # 3. Apply your moving average function
    def moving_average(values, window):
        weights = np.repeat(1.0, window) / window
        return np.convolve(values, weights, "valid")

    smooth_data = moving_average(total_energy, window_size)
    x_raw = np.arange(total_energy.shape[0])
    x_smooth = np.arange(window_size // 2, window_size // 2 + smooth_data.shape[0])

    fig = go.Figure()

    # Raw Total Energy
    fig.add_trace(
        go.Scatter(
            x=x_raw,
            y=total_energy,
            mode="lines",
            line=dict(width=1, color="rgba(0, 100, 255, 0.25)"),
            name="Raw Total Energy",
        )
    )

    # Smoothed Total Energy
    fig.add_trace(
        go.Scatter(
            x=x_smooth,
            y=smooth_data,
            mode="lines",
            line=dict(width=2.5, color="rgb(200, 20, 20)"),
            name=f"Moving Average (w={window_size})",
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title="Step / Time",
        yaxis_title="Total Potential Energy",
        template="plotly_white",
        hovermode="x unified",
    )
    fig.show()

In [210]:
plot_total_system_energy(X_uniform, title="Uniform Case")

plot_total_system_energy(X_stiffness, title="Stiffness Spread Case")

plot_total_system_energy(X_ratio, title="Ratio Spread Case")

In [211]:
def plot_lattice_heatmap(X_data, title="Lattice Energy Wave Propagation"):
    energy_matrix = 0.5 * (X_data**2)

    fig = go.Figure(
        data=go.Heatmap(
            z=energy_matrix.T,  # Transpose so nodes are on Y-axis and time is on X-axis
            colorscale="Viridis",  # Options: 'Viridis', 'Plasma', 'Inferno', 'Jet'
            colorbar=dict(title="Potential Energy"),
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title="Step / Time",
        yaxis_title="Node Index",
        template="plotly_white",
    )
    fig.show()

In [212]:
plot_lattice_heatmap(X_uniform, title="Uniform Case")

plot_lattice_heatmap(X_stiffness, title="Stiffness Spread Case")

plot_lattice_heatmap(X_ratio, title="Ratio Spread Case")